# ShelbyFirst ML Scenario Simulator

Peer-informed scenario simulation for ShelbyFirst built on the saved outputs from `final_model.ipynb`.

This notebook is descriptive and predictive, not causal. It uses similar-peer benchmarks and regularized ML models to simulate plausible shifts in:

- dental visit gap
- routine checkup gap
- obesity
- diabetes


### Cell 1 — Config / imports / paths


In [12]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Optional, Tuple
import json
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, ElasticNetCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
warnings.filterwarnings("ignore", message="Objective did not converge")

ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd() / "src",
    Path.cwd().parent,
    Path.cwd().parent / "src",
]


def find_project_root(candidates: List[Path]) -> Path:
    needed = [
        Path("data/processed/cluster_period_summary.parquet"),
        Path("data/processed/tract_cluster_map.parquet"),
        Path("data/processed/peer_similarity_transport/peer_model_base.parquet"),
    ]
    for root in candidates:
        if all((root / rel).exists() for rel in needed):
            return root
    searched = [str(p.resolve()) for p in candidates]
    raise FileNotFoundError(
        "Could not resolve the notebook data root. "
        f"Tried: {searched}"
    )


PROJECT_ROOT = find_project_root(ROOT_CANDIDATES)
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
OUT_DIR = DATA_DIR / "processed"
PEER_DIR = OUT_DIR / "peer_similarity_transport"
PRESENTATION_DIR = OUT_DIR / "presentation_ready"
ML_SIM_DIR = OUT_DIR / "ml_simulation"
ML_SIM_DIR.mkdir(parents=True, exist_ok=True)

CLUSTER_PERIOD_SUMMARY_PATH = OUT_DIR / "cluster_period_summary.parquet"
TRACT_CLUSTER_MAP_PATH = OUT_DIR / "tract_cluster_map.parquet"
PEER_BASE_PATH = PEER_DIR / "peer_model_base.parquet"
PEER_TOP10_PATH = PEER_DIR / "peer_similarity_transport_top10.csv"
SHELBY_PEER_TABLE_PATH = PEER_DIR / "shelby_peer_transport_table.csv"

ANALYSIS_PANEL_CANDIDATES = [
    OUT_DIR / "eda_panel_clean.parquet",
    OUT_DIR / "igs_x_acs_full.parquet",
]

HEALTH_LONG_CANDIDATES = [
    OUT_DIR / "cdc_places_cluster_tract_long.parquet",
    OUT_DIR / "cdc_places_tract_long.parquet",
    OUT_DIR / "cdc_places_shelby_tract_long.parquet",
    PRESENTATION_DIR / "cdc_places_shelby_tract_long.parquet",
]

HEALTH_WIDE_CANDIDATES = [
    OUT_DIR / "shelby_places_tract_base_expanded.parquet",
    OUT_DIR / "shelby_places_tract_base.parquet",
    PRESENTATION_DIR / "shelby_places_tract_base.parquet",
]

CDC_FETCH_CACHE = OUT_DIR / "cdc_places_cluster_tract_long.parquet"
CDC_FETCH_URL = "https://data.cdc.gov/resource/cwsq-ngmh.json"
CDC_MEASURE_IDS = ["DENTAL", "CHECKUP", "OBESITY", "DIABETES"]

RECOVERY_YEARS = [2022, 2023, 2024]
PRIMARY_TARGETS = ["dental_visit_gap", "routine_checkup_gap"]
SECONDARY_TARGETS = ["obesity_rate", "diabetes_rate"]
ALL_TARGETS = PRIMARY_TARGETS + SECONDARY_TARGETS

SCENARIO_STEPS = {
    "baseline": 0.00,
    "conservative": 0.15,
    "moderate": 0.30,
    "aggressive": 0.45,
}

TRACT_HEALTH_TARGETS_OUT = ML_SIM_DIR / "tract_health_targets.parquet"
CLUSTER_HEALTH_TARGETS_OUT = ML_SIM_DIR / "cluster_health_targets.parquet"
CLUSTER_ML_SIM_TABLE_OUT = ML_SIM_DIR / "cluster_ml_sim_table.parquet"
STAGE1_METRICS_OUT = ML_SIM_DIR / "stage1_model_metrics.csv"
STAGE2_METRICS_OUT = ML_SIM_DIR / "stage2_model_metrics.csv"
SHELBY_SCENARIO_FORECAST_OUT = ML_SIM_DIR / "shelby_scenario_forecast.csv"
SHELBY_SCENARIO_RANGES_OUT = ML_SIM_DIR / "shelby_scenario_ranges.csv"
SHELBY_SCENARIO_PRESENTATION_OUT = ML_SIM_DIR / "shelby_scenario_presentation_table.csv"
ACCESS_PLOT_OUT = ML_SIM_DIR / "shelby_scenario_access_gaps.png"
OUTCOME_PLOT_OUT = ML_SIM_DIR / "shelby_scenario_downstream_outcomes.png"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUT_DIR:", OUT_DIR)
print("ML_SIM_DIR:", ML_SIM_DIR)


PROJECT_ROOT: C:\Users\jabba\Desktop\Code\machine_learning\AUC_mastercard_challenge\src
OUT_DIR: C:\Users\jabba\Desktop\Code\machine_learning\AUC_mastercard_challenge\src\data\processed
ML_SIM_DIR: C:\Users\jabba\Desktop\Code\machine_learning\AUC_mastercard_challenge\src\data\processed\ml_simulation


### Cell 2 — Load main-notebook outputs


In [13]:
def require_columns(df: pd.DataFrame, cols: List[str], name: str) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")


def first_existing_path(paths: List[Path]) -> Optional[Path]:
    for path in paths:
        if path.exists():
            return path
    return None


def read_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported table format: {path}")


def normalize_geoid(values: pd.Series) -> pd.Series:
    s = values.astype("string").str.strip()
    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.str.replace(r"\D", "", regex=True)
    s = s.str.zfill(11)
    return s.where(s.str.fullmatch(r"\d{11}"), pd.NA)


def load_analysis_panel() -> pd.DataFrame:
    path = first_existing_path(ANALYSIS_PANEL_CANDIDATES)
    if path is None:
        searched = [str(p) for p in ANALYSIS_PANEL_CANDIDATES]
        raise FileNotFoundError(
            "Could not find the tract-year analysis panel. "
            f"Tried: {searched}"
        )
    df = pd.read_parquet(path).copy()
    require_columns(df, ["geoid", "year", "pop_total"], "analysis_panel")
    df["geoid"] = normalize_geoid(df["geoid"])
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    print("Loaded analysis panel from:", path)
    return df


def choose_shelby_cluster_id(cluster_period_summary_local: pd.DataFrame) -> str:
    exact = "Tennessee | Shelby County | cluster_77"
    if exact in set(cluster_period_summary_local["cluster_id"].astype(str)):
        return exact

    shelby_rows = cluster_period_summary_local.loc[
        (cluster_period_summary_local["display_state"].astype(str) == "Tennessee")
        & (cluster_period_summary_local["display_county"].astype(str) == "Shelby County")
    ].copy()

    if shelby_rows.empty:
        raise ValueError("Could not find Shelby County cluster in cluster_period_summary.")

    shelby_rows = shelby_rows.sort_values("igs_total_recovery", ascending=True)
    return str(shelby_rows.iloc[0]["cluster_id"])


cluster_period_summary = pd.read_parquet(CLUSTER_PERIOD_SUMMARY_PATH).copy()
tract_cluster_map = pd.read_parquet(TRACT_CLUSTER_MAP_PATH).copy()
peer_model_base = pd.read_parquet(PEER_BASE_PATH).copy()
peer_transport_top10 = pd.read_csv(PEER_TOP10_PATH).copy()
shelby_peer_transport_table = pd.read_csv(SHELBY_PEER_TABLE_PATH).copy()
analysis_panel = load_analysis_panel()

tract_cluster_map["geoid"] = normalize_geoid(tract_cluster_map["geoid"])
analysis_panel["geoid"] = normalize_geoid(analysis_panel["geoid"])

TARGET_CLUSTER_ID = choose_shelby_cluster_id(cluster_period_summary)

print("cluster_period_summary shape:", cluster_period_summary.shape)
print("tract_cluster_map shape:", tract_cluster_map.shape)
print("peer_model_base shape:", peer_model_base.shape)
print("peer_transport_top10 shape:", peer_transport_top10.shape)
print("Shelby cluster ID:", TARGET_CLUSTER_ID)
print()
print(
    cluster_period_summary.loc[
        cluster_period_summary["cluster_id"].astype(str) == TARGET_CLUSTER_ID,
        ["cluster_id", "display_state", "display_county", "n_cluster_tracts", "latest_cluster_pop_total", "igs_total_recovery"],
    ]
)


Loaded analysis panel from: C:\Users\jabba\Desktop\Code\machine_learning\AUC_mastercard_challenge\src\data\processed\eda_panel_clean.parquet
cluster_period_summary shape: (86, 48)
tract_cluster_map shape: (2942, 6)
peer_model_base shape: (86, 65)
peer_transport_top10 shape: (10, 59)
Shelby cluster ID: Tennessee | Shelby County | cluster_77

                                cluster_id display_state display_county  n_cluster_tracts  latest_cluster_pop_total  igs_total_recovery
76  Tennessee | Shelby County | cluster_77     Tennessee  Shelby County                96                  268298.0           35.813056


### Cell 3 — Define features and targets


In [14]:
recovery_features = [
    "igs_total_recovery",
    "igs_economy_recovery",
    "low_igs_tract_share_recovery",
    "poverty_rate_recovery",
    "unemp_rate_recovery",
    "median_household_income_recovery",
    "lfpr_16p_recovery",
]

trajectory_features = [
    "igs_total_drop_pre_to_covid",
    "igs_economy_drop_pre_to_covid",
    "poverty_increase_pre_to_covid",
    "unemp_increase_pre_to_covid",
    "income_drop_pre_to_covid",
    "lfpr_drop_pre_to_covid",
    "igs_total_rebound_covid_to_recovery",
    "igs_economy_rebound_covid_to_recovery",
    "poverty_improve_covid_to_recovery",
    "unemp_improve_covid_to_recovery",
    "income_rebound_covid_to_recovery",
    "lfpr_rebound_covid_to_recovery",
]

size_features = [
    "log_latest_cluster_pop_total",
    "log_n_cluster_tracts",
]

transport_features = [
    "no_vehicle_share_recovery",
    "one_vehicle_share_recovery",
    "transit_stops_per_1000_2024",
    "transit_stops_per_sqmi_2024",
]

raw_stage1_features = recovery_features + trajectory_features + size_features + transport_features


def usable_feature_columns(df: pd.DataFrame, features: List[str], min_nonmissing: int = 15) -> List[str]:
    usable = []
    for col in features:
        if col not in df.columns:
            continue
        nonmissing = int(df[col].notna().sum())
        nunique = int(df[col].nunique(dropna=True))
        if nonmissing >= min_nonmissing and nunique >= 2:
            usable.append(col)
    return usable


stage1_features = usable_feature_columns(peer_model_base, raw_stage1_features)
stage2_features = stage1_features + PRIMARY_TARGETS

if not set(recovery_features + trajectory_features + size_features).issubset(set(peer_model_base.columns)):
    require_columns(
        peer_model_base,
        recovery_features + trajectory_features + size_features,
        "peer_model_base",
    )

health_target_metadata = {
    "dental_visit_rate": {
        "measure_ids": ["DENTAL"],
        "wide_aliases": ["dental_visit_pct", "dental_visit_rate", "dental_visit"],
        "label_aliases": ["dental visit", "visited dentist", "dental clinic"],
    },
    "routine_checkup_rate": {
        "measure_ids": ["CHECKUP"],
        "wide_aliases": ["routine_checkup_pct", "routine_checkup_rate", "routine_checkup"],
        "label_aliases": ["routine checkup", "annual checkup", "visits to doctor"],
    },
    "obesity_rate": {
        "measure_ids": ["OBESITY"],
        "wide_aliases": ["obesity_pct", "obesity_rate", "obesity"],
        "label_aliases": ["obesity"],
    },
    "diabetes_rate": {
        "measure_ids": ["DIABETES"],
        "wide_aliases": ["diabetes_pct", "diabetes_rate", "diabetes"],
        "label_aliases": ["diabetes"],
    },
}

print("Stage 1 features used:", stage1_features)
print("Primary targets:", PRIMARY_TARGETS)
print("Secondary targets:", SECONDARY_TARGETS)


Stage 1 features used: ['igs_total_recovery', 'igs_economy_recovery', 'low_igs_tract_share_recovery', 'poverty_rate_recovery', 'unemp_rate_recovery', 'median_household_income_recovery', 'lfpr_16p_recovery', 'igs_total_drop_pre_to_covid', 'igs_economy_drop_pre_to_covid', 'poverty_increase_pre_to_covid', 'unemp_increase_pre_to_covid', 'income_drop_pre_to_covid', 'lfpr_drop_pre_to_covid', 'igs_total_rebound_covid_to_recovery', 'igs_economy_rebound_covid_to_recovery', 'poverty_improve_covid_to_recovery', 'unemp_improve_covid_to_recovery', 'income_rebound_covid_to_recovery', 'lfpr_rebound_covid_to_recovery', 'log_latest_cluster_pop_total', 'log_n_cluster_tracts', 'no_vehicle_share_recovery', 'one_vehicle_share_recovery']
Primary targets: ['dental_visit_gap', 'routine_checkup_gap']
Secondary targets: ['obesity_rate', 'diabetes_rate']


### Cell 4 — Load tract-level health targets


In [15]:
def detect_geoid_column(df: pd.DataFrame) -> str:
    candidates = [
        "geoid",
        "tractfips",
        "locationid",
        "locationname",
        "GEOID",
    ]
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    raise ValueError(
        "Could not detect a tract GEOID column in the health input. "
        f"Available columns: {list(df.columns)}"
    )


def detect_long_columns(df: pd.DataFrame) -> Tuple[Optional[str], Optional[str], Optional[str]]:
    indicator_candidates = ["measureid", "measure", "short_question_text"]
    value_candidates = ["data_value", "value", "estimate"]
    year_candidates = ["year", "Year"]

    indicator_col = next((c for c in indicator_candidates if c in df.columns), None)
    value_col = next((c for c in value_candidates if c in df.columns), None)
    year_col = next((c for c in year_candidates if c in df.columns), None)
    return indicator_col, value_col, year_col


def first_matching_column(df: pd.DataFrame, aliases: List[str]) -> Optional[str]:
    lower_map = {c.lower(): c for c in df.columns}
    for alias in aliases:
        if alias.lower() in lower_map:
            return lower_map[alias.lower()]
    for col in df.columns:
        col_lower = col.lower()
        if any(alias.lower() in col_lower for alias in aliases):
            return col
    return None


def build_targets_from_wide(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    geoid_col = detect_geoid_column(df)
    out = pd.DataFrame({"geoid": normalize_geoid(df[geoid_col])})
    out = out.dropna(subset=["geoid"]).copy()

    for canonical, meta in health_target_metadata.items():
        src_col = first_matching_column(df, meta["wide_aliases"])
        if src_col is not None:
            out[canonical] = pd.to_numeric(df[src_col], errors="coerce")

    if "dental_visit_rate" in out.columns:
        out["dental_visit_gap"] = 100 - out["dental_visit_rate"]
    if "routine_checkup_rate" in out.columns:
        out["routine_checkup_gap"] = 100 - out["routine_checkup_rate"]

    for col in ["dental_visit_gap", "routine_checkup_gap", "obesity_rate", "diabetes_rate"]:
        if col not in out.columns:
            out[col] = np.nan

    out["wide_source"] = source_name
    keep_cols = ["geoid", "dental_visit_gap", "routine_checkup_gap", "obesity_rate", "diabetes_rate", "wide_source"]
    return out[keep_cols].drop_duplicates(subset=["geoid"]).reset_index(drop=True)


def build_targets_from_long(df: pd.DataFrame, source_name: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    geoid_col = detect_geoid_column(df)
    indicator_col, value_col, year_col = detect_long_columns(df)

    if indicator_col is None or value_col is None:
        raise ValueError(
            "The long-format health file is missing the indicator or value column. "
            f"Columns found: {list(df.columns)}"
        )

    long_df = df.copy()
    long_df["geoid"] = normalize_geoid(long_df[geoid_col])
    long_df[value_col] = pd.to_numeric(long_df[value_col], errors="coerce")
    long_df = long_df.dropna(subset=["geoid", value_col]).copy()

    if year_col is not None:
        long_df[year_col] = pd.to_numeric(long_df[year_col], errors="coerce")

    measure_text = (
        long_df.get("measureid", pd.Series(index=long_df.index, dtype="string")).astype("string").fillna("")
        + " | "
        + long_df.get("measure", pd.Series(index=long_df.index, dtype="string")).astype("string").fillna("")
        + " | "
        + long_df.get("short_question_text", pd.Series(index=long_df.index, dtype="string")).astype("string").fillna("")
    ).str.lower()

    pieces = []
    audit_rows = []

    for canonical, meta in health_target_metadata.items():
        mask = pd.Series(False, index=long_df.index)
        if "measureid" in long_df.columns:
            mask = mask | long_df["measureid"].astype("string").isin(meta["measure_ids"])
        for alias in meta["label_aliases"]:
            mask = mask | measure_text.str.contains(alias.lower(), regex=False, na=False)

        sub = long_df.loc[mask].copy()
        if sub.empty:
            continue

        if year_col is not None and sub[year_col].notna().any():
            latest_year = int(sub[year_col].dropna().max())
            sub = sub.loc[sub[year_col] == latest_year].copy()
        else:
            latest_year = np.nan

        wide_piece = (
            sub.groupby("geoid", as_index=False)[value_col]
            .mean()
            .rename(columns={value_col: canonical})
        )
        pieces.append(wide_piece)
        audit_rows.append(
            {
                "target_name": canonical,
                "source_name": source_name,
                "latest_year_used": latest_year,
                "n_geoids": wide_piece["geoid"].nunique(),
            }
        )

    if not pieces:
        raise ValueError(
            "Could not map any requested targets from the long-format health file. "
            f"Columns available: {list(df.columns)}"
        )

    out = pieces[0]
    for piece in pieces[1:]:
        out = out.merge(piece, on="geoid", how="outer")

    if "dental_visit_rate" in out.columns:
        out["dental_visit_gap"] = 100 - out["dental_visit_rate"]
    if "routine_checkup_rate" in out.columns:
        out["routine_checkup_gap"] = 100 - out["routine_checkup_rate"]

    for col in ["dental_visit_gap", "routine_checkup_gap", "obesity_rate", "diabetes_rate"]:
        if col not in out.columns:
            out[col] = np.nan

    out["long_source"] = source_name
    audit = pd.DataFrame(audit_rows)
    keep_cols = ["geoid", "dental_visit_gap", "routine_checkup_gap", "obesity_rate", "diabetes_rate", "long_source"]
    return out[keep_cols].drop_duplicates(subset=["geoid"]).reset_index(drop=True), audit


def fetch_cdc_places_cluster_tract_long(geoids: List[str], out_path: Path, chunk_size: int = 150) -> pd.DataFrame:
    session = requests.Session()
    session.headers.update({"User-Agent": "shelbyfirst-ml-simulator/1.0"})
    select_cols = ",".join(
        [
            "year",
            "stateabbr",
            "statedesc",
            "countyname",
            "countyfips",
            "locationname",
            "datasource",
            "category",
            "measure",
            "data_value_unit",
            "data_value_type",
            "data_value",
            "low_confidence_limit",
            "high_confidence_limit",
            "totalpopulation",
            "totalpop18plus",
            "geolocation",
            "locationid",
            "categoryid",
            "measureid",
            "datavaluetypeid",
            "short_question_text",
        ]
    )

    chunks = [geoids[i:i + chunk_size] for i in range(0, len(geoids), chunk_size)]
    frames = []

    for idx, chunk in enumerate(chunks, start=1):
        where_geoids = ",".join(f"'{g}'" for g in chunk)
        where_measures = ",".join(f"'{m}'" for m in CDC_MEASURE_IDS)
        params = {
            "$select": select_cols,
            "$where": f"locationid in ({where_geoids}) AND measureid in ({where_measures})",
            "$limit": 50000,
        }
        response = session.get(CDC_FETCH_URL, params=params, timeout=120)
        response.raise_for_status()
        rows = response.json()
        print(f"CDC chunk {idx}/{len(chunks)} returned {len(rows)} rows")
        if rows:
            frame = pd.DataFrame(rows)
            if "geolocation" in frame.columns:
                frame["geolocation"] = frame["geolocation"].map(
                    lambda x: json.dumps(x, sort_keys=True) if isinstance(x, dict) else x
                )
            frames.append(frame)

    if not frames:
        raise ValueError("CDC PLACES fetch returned zero rows for the tract cluster universe.")

    out = pd.concat(frames, ignore_index=True).drop_duplicates().copy()
    out["tractfips"] = normalize_geoid(out["locationid"])
    out.to_parquet(out_path, index=False)
    return out


long_health_path = first_existing_path(HEALTH_LONG_CANDIDATES)
wide_health_path = first_existing_path(HEALTH_WIDE_CANDIDATES)

if long_health_path is None and CDC_FETCH_CACHE.exists():
    long_health_path = CDC_FETCH_CACHE

if long_health_path is None:
    print("No tract-level long health cache found. Fetching CDC PLACES tract rows for cluster universe...")
    needed_geoids = sorted(tract_cluster_map["geoid"].dropna().astype(str).unique().tolist())
    fetched = fetch_cdc_places_cluster_tract_long(needed_geoids, CDC_FETCH_CACHE)
    long_health_path = CDC_FETCH_CACHE
    print("Fetched and cached CDC PLACES rows:", fetched.shape)

health_long_audit = pd.DataFrame()
long_targets = None
if long_health_path is not None:
    health_long_raw = read_table(long_health_path)
    long_targets, health_long_audit = build_targets_from_long(
        health_long_raw,
        source_name=str(long_health_path.relative_to(PROJECT_ROOT)),
    )
    print("Loaded long-format health source:", long_health_path)
    print("health_long_raw shape:", health_long_raw.shape)
    print(health_long_audit)

wide_targets = None
if wide_health_path is not None:
    health_wide_raw = read_table(wide_health_path)
    wide_targets = build_targets_from_wide(
        health_wide_raw,
        source_name=str(wide_health_path.relative_to(PROJECT_ROOT)),
    )
    print("Loaded wide-format health source:", wide_health_path)
    print("health_wide_raw shape:", health_wide_raw.shape)

if long_targets is None and wide_targets is None:
    raise ValueError("No usable health target source was found.")

if long_targets is not None and wide_targets is not None:
    tract_health_targets = long_targets.merge(wide_targets, on="geoid", how="outer", suffixes=("_long", "_wide"))
    for col in ALL_TARGETS:
        tract_health_targets[col] = tract_health_targets.get(f"{col}_long").combine_first(
            tract_health_targets.get(f"{col}_wide")
        )
    tract_health_targets["target_source"] = np.where(
        tract_health_targets["dental_visit_gap_long"].notna()
        | tract_health_targets["routine_checkup_gap_long"].notna()
        | tract_health_targets["obesity_rate_long"].notna()
        | tract_health_targets["diabetes_rate_long"].notna(),
        "long_primary_with_wide_backfill",
        "wide_only",
    )
    tract_health_targets = tract_health_targets[["geoid"] + ALL_TARGETS + ["target_source"]].copy()
elif long_targets is not None:
    tract_health_targets = long_targets.copy()
    tract_health_targets["target_source"] = "long_only"
else:
    tract_health_targets = wide_targets.copy()
    tract_health_targets["target_source"] = "wide_only"

tract_health_targets = tract_health_targets.drop_duplicates(subset=["geoid"]).sort_values("geoid").reset_index(drop=True)
tract_health_targets.to_parquet(TRACT_HEALTH_TARGETS_OUT, index=False)

print("tract_health_targets shape:", tract_health_targets.shape)
print("tract_health_targets missing share:")
print(tract_health_targets[ALL_TARGETS].isna().mean().sort_values())
print()
print(tract_health_targets.head())


Loaded long-format health source: C:\Users\jabba\Desktop\Code\machine_learning\AUC_mastercard_challenge\src\data\processed\cdc_places_cluster_tract_long.parquet
health_long_raw shape: (11154, 23)
            target_name                                        source_name  latest_year_used  n_geoids
0     dental_visit_rate  data\processed\cdc_places_cluster_tract_long.p...              2022      2868
1  routine_checkup_rate  data\processed\cdc_places_cluster_tract_long.p...              2023      2762
2          obesity_rate  data\processed\cdc_places_cluster_tract_long.p...              2023      2762
3         diabetes_rate  data\processed\cdc_places_cluster_tract_long.p...              2023      2762
Loaded wide-format health source: C:\Users\jabba\Desktop\Code\machine_learning\AUC_mastercard_challenge\src\data\processed\shelby_places_tract_base_expanded.parquet
health_wide_raw shape: (249, 28)
tract_health_targets shape: (3024, 6)
tract_health_targets missing share:
dental_visit_gap 

### Cell 5 — Aggregate tract health targets to clusters


In [16]:
analysis_panel["year"] = pd.to_numeric(analysis_panel["year"], errors="coerce")
tract_recovery_weights = (
    analysis_panel.loc[analysis_panel["year"].isin(RECOVERY_YEARS), ["geoid", "pop_total"]]
    .copy()
    .assign(pop_total=lambda df: pd.to_numeric(df["pop_total"], errors="coerce"))
    .groupby("geoid", as_index=False)["pop_total"]
    .mean()
    .rename(columns={"pop_total": "recovery_pop"})
)


def weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    vals = pd.to_numeric(values, errors="coerce")
    wts = pd.to_numeric(weights, errors="coerce")
    mask = vals.notna() & wts.notna() & (wts > 0)
    if mask.any():
        return float(np.average(vals.loc[mask], weights=wts.loc[mask]))
    if vals.notna().any():
        return float(vals.dropna().mean())
    return np.nan


tract_cluster_health = (
    tract_cluster_map
    .merge(tract_recovery_weights, on="geoid", how="left")
    .merge(tract_health_targets, on="geoid", how="left")
)

cluster_rows = []
for cluster_id, grp in tract_cluster_health.groupby("cluster_id", dropna=False):
    row = {
        "cluster_id": cluster_id,
        "display_state": grp["display_state"].dropna().iloc[0] if grp["display_state"].notna().any() else pd.NA,
        "display_county": grp["display_county"].dropna().iloc[0] if grp["display_county"].notna().any() else pd.NA,
        "n_cluster_tracts": int(grp["geoid"].nunique()),
        "n_health_matched_tracts_any": int(grp[ALL_TARGETS].notna().any(axis=1).sum()),
        "health_match_share_any": float(grp[ALL_TARGETS].notna().any(axis=1).mean()) if len(grp) else np.nan,
    }

    for target in ALL_TARGETS:
        row[f"n_{target}_tracts"] = int(grp[target].notna().sum())
        row[f"share_{target}_tracts"] = float(grp[target].notna().mean()) if len(grp) else np.nan
        row[target] = weighted_mean(grp[target], grp["recovery_pop"])

    cluster_rows.append(row)

cluster_health_targets = (
    pd.DataFrame(cluster_rows)
    .sort_values(["display_state", "display_county", "cluster_id"])
    .reset_index(drop=True)
)

cluster_health_targets["health_targets_complete_flag"] = cluster_health_targets[ALL_TARGETS].notna().all(axis=1)
cluster_health_targets.to_parquet(CLUSTER_HEALTH_TARGETS_OUT, index=False)

print("tract_cluster_health shape:", tract_cluster_health.shape)
print("cluster_health_targets shape:", cluster_health_targets.shape)
print("Cluster nonmissing target share:")
print(cluster_health_targets[ALL_TARGETS].notna().mean().sort_values())
print()
print("Shelby cluster health targets:")
print(
    cluster_health_targets.loc[
        cluster_health_targets["cluster_id"].astype(str) == TARGET_CLUSTER_ID,
        ["cluster_id", "n_cluster_tracts"] + ALL_TARGETS + [f"share_{t}_tracts" for t in ALL_TARGETS],
    ]
)


tract_cluster_health shape: (2942, 12)
cluster_health_targets shape: (86, 19)
Cluster nonmissing target share:
routine_checkup_gap    0.906977
obesity_rate           0.906977
diabetes_rate          0.906977
dental_visit_gap       0.965116
dtype: float64

Shelby cluster health targets:
                                cluster_id  n_cluster_tracts  dental_visit_gap  routine_checkup_gap  obesity_rate  diabetes_rate  share_dental_visit_gap_tracts  share_routine_checkup_gap_tracts  share_obesity_rate_tracts  \
76  Tennessee | Shelby County | cluster_77                96         55.913424            17.952099     46.168349      21.587828                        0.96875                           0.96875                    0.96875   

    share_diabetes_rate_tracts  
76                     0.96875  


### Cell 6 — Build final cluster ML simulation table


In [17]:
cluster_ml_sim_table = (
    peer_model_base
    .merge(cluster_health_targets, on=["cluster_id", "display_state", "display_county", "n_cluster_tracts"], how="left")
    .copy()
)

cluster_ml_sim_table["is_shelby_target"] = cluster_ml_sim_table["cluster_id"].astype(str) == TARGET_CLUSTER_ID
cluster_ml_sim_table["transport_peer_rank"] = cluster_ml_sim_table["cluster_id"].map(
    peer_transport_top10.set_index("cluster_id")["peer_rank"].to_dict()
)
cluster_ml_sim_table["is_top_transport_peer"] = cluster_ml_sim_table["transport_peer_rank"].notna()
cluster_ml_sim_table["health_complete_primary_flag"] = cluster_ml_sim_table[PRIMARY_TARGETS].notna().all(axis=1)
cluster_ml_sim_table["health_complete_secondary_flag"] = cluster_ml_sim_table[SECONDARY_TARGETS].notna().all(axis=1)
cluster_ml_sim_table["health_complete_all_flag"] = cluster_ml_sim_table[ALL_TARGETS].notna().all(axis=1)

cluster_ml_sim_table.to_parquet(CLUSTER_ML_SIM_TABLE_OUT, index=False)

print("cluster_ml_sim_table shape:", cluster_ml_sim_table.shape)
print("Missing share by target:")
print(cluster_ml_sim_table[ALL_TARGETS].isna().mean().sort_values())
print()
print(
    cluster_ml_sim_table.loc[
        cluster_ml_sim_table["cluster_id"].astype(str) == TARGET_CLUSTER_ID,
        ["cluster_id", "display_state", "display_county"] + ALL_TARGETS,
    ]
)


cluster_ml_sim_table shape: (86, 86)
Missing share by target:
dental_visit_gap       0.034884
routine_checkup_gap    0.093023
obesity_rate           0.093023
diabetes_rate          0.093023
dtype: float64

                                cluster_id display_state display_county  dental_visit_gap  routine_checkup_gap  obesity_rate  diabetes_rate
76  Tennessee | Shelby County | cluster_77     Tennessee  Shelby County         55.913424            17.952099     46.168349      21.587828


### Cell 7 — Exploratory checks


In [18]:
print("Target availability counts:")
print(cluster_ml_sim_table[ALL_TARGETS].notna().sum().sort_values())
print()

shelby_row = cluster_ml_sim_table.loc[cluster_ml_sim_table["cluster_id"].astype(str) == TARGET_CLUSTER_ID].copy()
if shelby_row.empty:
    raise ValueError(f"Could not find Shelby row in cluster_ml_sim_table: {TARGET_CLUSTER_ID}")

print("Shelby baseline row:")
print(
    shelby_row[
        [
            "cluster_id",
            "display_state",
            "display_county",
            "n_cluster_tracts",
            "latest_cluster_pop_total",
        ] + stage1_features + ALL_TARGETS
    ].T
)
print()

corr_cols = [c for c in stage1_features + ALL_TARGETS if c in cluster_ml_sim_table.columns]
corr_df = cluster_ml_sim_table[corr_cols].apply(pd.to_numeric, errors="coerce")
print("Correlation snapshot:")
print(corr_df.corr(numeric_only=True).loc[ALL_TARGETS, stage1_features[:12]].round(3))


Target availability counts:
routine_checkup_gap    78
obesity_rate           78
diabetes_rate          78
dental_visit_gap       83
dtype: int64

Shelby baseline row:
                                                                           76
cluster_id                             Tennessee | Shelby County | cluster_77
display_state                                                       Tennessee
display_county                                                  Shelby County
n_cluster_tracts                                                           96
latest_cluster_pop_total                                             268298.0
igs_total_recovery                                                  35.813056
igs_economy_recovery                                                35.120633
low_igs_tract_share_recovery                                         0.896057
poverty_rate_recovery                                                0.300348
unemp_rate_recovery                                  

### Cell 8 — Modeling helpers


In [19]:
def make_cv(n_samples: int) -> KFold:
    n_splits = min(5, max(3, n_samples // 12))
    return KFold(n_splits=n_splits, shuffle=True, random_state=42)


def make_elastic_net_cv(cv) -> Pipeline:
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "model",
                ElasticNetCV(
                    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
                    alphas=np.logspace(-3, 2, 60),
                    cv=cv,
                    max_iter=20000,
                    random_state=42,
                ),
            ),
        ]
    )


def make_fixed_elastic_net(alpha: float, l1_ratio: float) -> Pipeline:
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "model",
                ElasticNet(
                    alpha=float(alpha),
                    l1_ratio=float(l1_ratio),
                    max_iter=20000,
                    random_state=42,
                ),
            ),
        ]
    )


def clip_pct(value: float) -> float:
    if pd.isna(value):
        return np.nan
    return float(np.clip(value, 0.0, 100.0))


def evaluate_model(df: pd.DataFrame, features: List[str], target: str, stage_name: str) -> Tuple[Dict[str, object], Pipeline, pd.DataFrame]:
    model_df = df[features + [target, "cluster_id", "display_state", "display_county"]].copy()
    model_df[target] = pd.to_numeric(model_df[target], errors="coerce")
    model_df = model_df.loc[model_df[target].notna()].copy()

    if len(model_df) < 25:
        raise ValueError(f"{stage_name} {target}: not enough rows to model ({len(model_df)}).")

    X = model_df[features].apply(pd.to_numeric, errors="coerce")
    y = model_df[target].astype(float)
    cv = make_cv(len(model_df))
    pipeline = make_elastic_net_cv(cv)

    y_pred_cv = cross_val_predict(pipeline, X, y, cv=cv)
    pipeline.fit(X, y)

    model = pipeline.named_steps["model"]
    metric_row = {
        "stage_name": stage_name,
        "target": target,
        "n_rows": len(model_df),
        "n_features": len(features),
        "cv_splits": cv.get_n_splits(),
        "rmse": float(np.sqrt(mean_squared_error(y, y_pred_cv))),
        "mae": float(mean_absolute_error(y, y_pred_cv)),
        "r2": float(r2_score(y, y_pred_cv)),
        "elastic_net_alpha": float(model.alpha_),
        "elastic_net_l1_ratio": float(model.l1_ratio_),
        "target_mean": float(y.mean()),
        "target_std": float(y.std(ddof=0)),
    }

    prediction_df = model_df[["cluster_id", "display_state", "display_county", target]].copy()
    prediction_df["cv_pred"] = y_pred_cv
    prediction_df["residual"] = prediction_df[target] - prediction_df["cv_pred"]
    prediction_df["stage_name"] = stage_name

    return metric_row, pipeline, prediction_df


def get_coefficient_table(pipeline: Pipeline, features: List[str], target: str) -> pd.DataFrame:
    model = pipeline.named_steps["model"]
    coef = pd.Series(model.coef_, index=features, name="coefficient")
    out = coef.reset_index().rename(columns={"index": "feature"})
    out["target"] = target
    out["abs_coefficient"] = out["coefficient"].abs()
    return out.sort_values("abs_coefficient", ascending=False).reset_index(drop=True)


### Cell 9 — Train Stage 1 access models


In [20]:
stage1_metrics = []
stage1_models: Dict[str, Pipeline] = {}
stage1_predictions = []
stage1_coefficients = []

stage1_df = cluster_ml_sim_table.copy()

for target in PRIMARY_TARGETS:
    metric_row, fitted_model, pred_df = evaluate_model(
        df=stage1_df,
        features=stage1_features,
        target=target,
        stage_name="stage1_access",
    )
    stage1_metrics.append(metric_row)
    stage1_models[target] = fitted_model
    stage1_predictions.append(pred_df)
    stage1_coefficients.append(get_coefficient_table(fitted_model, stage1_features, target))

stage1_model_metrics = pd.DataFrame(stage1_metrics).sort_values("target").reset_index(drop=True)
stage1_model_metrics.to_csv(STAGE1_METRICS_OUT, index=False)

stage1_predictions_df = pd.concat(stage1_predictions, ignore_index=True)
stage1_coefficients_df = pd.concat(stage1_coefficients, ignore_index=True)

print(stage1_model_metrics)
print()
print("Top Stage 1 coefficients:")
print(stage1_coefficients_df.groupby("target").head(8))


      stage_name               target  n_rows  n_features  cv_splits      rmse       mae        r2  elastic_net_alpha  elastic_net_l1_ratio  target_mean  target_std
0  stage1_access     dental_visit_gap      83          23          5  6.217570  4.011597  0.411595           0.348637                   0.9    49.388475    8.105552
1  stage1_access  routine_checkup_gap      78          23          5  4.914408  3.843571  0.033868           0.286832                   0.9    23.166647    4.999804

Top Stage 1 coefficients:
                                  feature  coefficient               target  abs_coefficient
0            low_igs_tract_share_recovery     3.432979     dental_visit_gap         3.432979
1        median_household_income_recovery    -2.878045     dental_visit_gap         2.878045
2                    igs_economy_recovery     1.450005     dental_visit_gap         1.450005
3           igs_economy_drop_pre_to_covid    -1.399293     dental_visit_gap         1.399293
4       pover

### Cell 10 — Train Stage 2 downstream health models


In [21]:
stage2_metrics = []
stage2_models: Dict[str, Pipeline] = {}
stage2_predictions = []
stage2_coefficients = []

stage2_df = cluster_ml_sim_table.copy()

for target in SECONDARY_TARGETS:
    metric_row, fitted_model, pred_df = evaluate_model(
        df=stage2_df,
        features=stage2_features,
        target=target,
        stage_name="stage2_downstream",
    )
    stage2_metrics.append(metric_row)
    stage2_models[target] = fitted_model
    stage2_predictions.append(pred_df)
    stage2_coefficients.append(get_coefficient_table(fitted_model, stage2_features, target))

stage2_model_metrics = pd.DataFrame(stage2_metrics).sort_values("target").reset_index(drop=True)
stage2_model_metrics.to_csv(STAGE2_METRICS_OUT, index=False)

stage2_predictions_df = pd.concat(stage2_predictions, ignore_index=True)
stage2_coefficients_df = pd.concat(stage2_coefficients, ignore_index=True)

print(stage2_model_metrics)
print()
print("Top Stage 2 coefficients:")
print(stage2_coefficients_df.groupby("target").head(10))


          stage_name         target  n_rows  n_features  cv_splits      rmse       mae        r2  elastic_net_alpha  elastic_net_l1_ratio  target_mean  target_std
0  stage2_downstream  diabetes_rate      78          25          5  2.323215  1.453636  0.469378           0.194149                   0.1    15.649949    3.189311
1  stage2_downstream   obesity_rate      78          25          5  5.014394  3.517396  0.486545           0.194149                   0.3    36.779279    6.997891

Top Stage 2 coefficients:
                                feature  coefficient         target  abs_coefficient
0             no_vehicle_share_recovery    -2.382940   obesity_rate         2.382940
1      median_household_income_recovery    -2.260749   obesity_rate         2.260749
2                   routine_checkup_gap    -2.098014   obesity_rate         2.098014
3                     lfpr_16p_recovery     1.727411   obesity_rate         1.727411
4                  log_n_cluster_tracts     1.218305   obes

### Cell 11 — Build Shelby peer benchmarks


In [22]:
shelby_row = cluster_ml_sim_table.loc[cluster_ml_sim_table["cluster_id"].astype(str) == TARGET_CLUSTER_ID].copy()
if shelby_row.empty:
    raise ValueError(f"Shelby target row missing from cluster_ml_sim_table: {TARGET_CLUSTER_ID}")

peer_benchmark_base = (
    peer_transport_top10[
        ["peer_rank", "cluster_id", "display_state", "display_county", "similarity_score"]
    ]
    .merge(
        cluster_ml_sim_table[["cluster_id"] + ALL_TARGETS],
        on="cluster_id",
        how="left",
        validate="1:1",
    )
    .sort_values("peer_rank")
    .reset_index(drop=True)
)

shelby_access = {target: float(shelby_row[target].iloc[0]) for target in PRIMARY_TARGETS}

benchmark_rows = []
for target in PRIMARY_TARGETS:
    better = peer_benchmark_base.loc[
        peer_benchmark_base[target].notna()
        & (peer_benchmark_base[target] < shelby_access[target])
    ].copy()

    if better.empty:
        better = peer_benchmark_base.loc[peer_benchmark_base[target].notna()].copy()

    better = better.head(5).copy()
    weights = better["similarity_score"].clip(lower=0).astype(float)
    if weights.sum() <= 0:
        benchmark_value = float(better[target].mean())
    else:
        benchmark_value = float(np.average(better[target], weights=weights))

    benchmark_rows.append(
        {
            "target": target,
            "shelby_value": shelby_access[target],
            "benchmark_value": benchmark_value,
            "improvement_available": float(shelby_access[target] - benchmark_value),
            "peer_count_used": int(len(better)),
            "peer_cluster_ids": " | ".join(better["cluster_id"].astype(str).tolist()),
        }
    )

shelby_peer_benchmarks = pd.DataFrame(benchmark_rows)

peer_benchmark_base["better_dental_peer"] = peer_benchmark_base["dental_visit_gap"] < shelby_access["dental_visit_gap"]
peer_benchmark_base["better_checkup_peer"] = peer_benchmark_base["routine_checkup_gap"] < shelby_access["routine_checkup_gap"]
peer_benchmark_base["better_access_peer"] = peer_benchmark_base["better_dental_peer"] | peer_benchmark_base["better_checkup_peer"]

print("Shelby peer benchmark inputs:")
print(peer_benchmark_base)
print()
print("Shelby peer benchmarks:")
print(shelby_peer_benchmarks)


Shelby peer benchmark inputs:
   peer_rank                                     cluster_id display_state      display_county  similarity_score  dental_visit_gap  routine_checkup_gap  obesity_rate  diabetes_rate  better_dental_peer  better_checkup_peer  \
0          1        Louisiana | Orleans Parish | cluster_33     Louisiana      Orleans Parish          0.942714         47.486551            18.383149     38.075851      18.219990                True                False   
1          2           Michigan | Wayne County | cluster_38      Michigan        Wayne County          0.924209         50.783732            17.596175     47.181600      20.459905                True                 True   
2          3      Wisconsin | Milwaukee County | cluster_86     Wisconsin    Milwaukee County          0.922566         52.986970            20.879191     47.877497      18.671161                True                False   
3          4            Ohio | Cuyahoga County | cluster_63          Ohio 

### Cell 12 — Define Shelby scenarios


In [23]:
shelby_feature_row = shelby_row.iloc[[0]].copy()
scenario_rows = []

benchmark_lookup = shelby_peer_benchmarks.set_index("target")["benchmark_value"].to_dict()

for scenario_name, step_share in SCENARIO_STEPS.items():
    row = shelby_feature_row.copy()
    row["scenario_name"] = scenario_name
    row["scenario_step_share"] = step_share

    for target in PRIMARY_TARGETS:
        baseline_value = float(shelby_feature_row[target].iloc[0])
        benchmark_value = float(benchmark_lookup[target])
        scenario_value = baseline_value + step_share * (benchmark_value - baseline_value)
        row[target] = clip_pct(scenario_value)
        row[f"{target}_baseline"] = baseline_value
        row[f"{target}_benchmark"] = benchmark_value
        row[f"{target}_improvement_pct_points"] = baseline_value - float(row[target].iloc[0])

    scenario_rows.append(row)

shelby_scenarios = pd.concat(scenario_rows, ignore_index=True)

baseline_stage1_predictions = {
    target: clip_pct(stage1_models[target].predict(shelby_feature_row[stage1_features])[0])
    for target in PRIMARY_TARGETS
}

print("Baseline Shelby access values:", shelby_access)
print("Stage 1 model baseline predictions:", baseline_stage1_predictions)
print()
print(
    shelby_scenarios[
        [
            "scenario_name",
            "scenario_step_share",
            "dental_visit_gap",
            "routine_checkup_gap",
            "dental_visit_gap_improvement_pct_points",
            "routine_checkup_gap_improvement_pct_points",
        ]
    ]
)


Baseline Shelby access values: {'dental_visit_gap': 55.91342417197462, 'routine_checkup_gap': 17.95209894101547}
Stage 1 model baseline predictions: {'dental_visit_gap': 54.27028541188597, 'routine_checkup_gap': 21.49054190053495}

  scenario_name  scenario_step_share  dental_visit_gap  routine_checkup_gap  dental_visit_gap_improvement_pct_points  routine_checkup_gap_improvement_pct_points
0      baseline                 0.00         55.913424            17.952099                                 0.000000                                    0.000000
1  conservative                 0.15         55.211568            17.899575                                 0.701856                                    0.052524
2      moderate                 0.30         54.509712            17.847051                                 1.403712                                    0.105048
3    aggressive                 0.45         53.807856            17.794527                                 2.105569        

### Cell 13 — Forecast Shelby under each scenario


In [24]:
observed_baseline = {
    target: float(shelby_row[target].iloc[0])
    for target in ALL_TARGETS
}

baseline_stage2_features = pd.DataFrame([shelby_feature_row.iloc[0][stage2_features]])
baseline_model_predictions = {
    target: clip_pct(stage2_models[target].predict(baseline_stage2_features)[0])
    for target in SECONDARY_TARGETS
}

forecast_rows = []
for _, scenario_row in shelby_scenarios.iterrows():
    scenario_name = scenario_row["scenario_name"]
    scenario_features = pd.DataFrame([scenario_row[stage2_features]])

    obesity_model_pred = clip_pct(stage2_models["obesity_rate"].predict(scenario_features)[0])
    diabetes_model_pred = clip_pct(stage2_models["diabetes_rate"].predict(scenario_features)[0])

    obesity_projected = clip_pct(
        observed_baseline["obesity_rate"] + (obesity_model_pred - baseline_model_predictions["obesity_rate"])
    )
    diabetes_projected = clip_pct(
        observed_baseline["diabetes_rate"] + (diabetes_model_pred - baseline_model_predictions["diabetes_rate"])
    )

    forecast_rows.append(
        {
            "scenario_name": scenario_name,
            "scenario_step_share": float(scenario_row["scenario_step_share"]),
            "cluster_id": TARGET_CLUSTER_ID,
            "dental_visit_gap_pct": float(scenario_row["dental_visit_gap"]),
            "routine_checkup_gap_pct": float(scenario_row["routine_checkup_gap"]),
            "obesity_pct_projected": obesity_projected,
            "diabetes_pct_projected": diabetes_projected,
            "obesity_pct_baseline_observed": observed_baseline["obesity_rate"],
            "diabetes_pct_baseline_observed": observed_baseline["diabetes_rate"],
            "dental_visit_gap_pct_baseline_observed": observed_baseline["dental_visit_gap"],
            "routine_checkup_gap_pct_baseline_observed": observed_baseline["routine_checkup_gap"],
            "obesity_pct_model_baseline": baseline_model_predictions["obesity_rate"],
            "diabetes_pct_model_baseline": baseline_model_predictions["diabetes_rate"],
            "obesity_pct_delta_vs_observed": obesity_projected - observed_baseline["obesity_rate"],
            "diabetes_pct_delta_vs_observed": diabetes_projected - observed_baseline["diabetes_rate"],
            "dental_visit_gap_pct_delta_vs_observed": float(scenario_row["dental_visit_gap"]) - observed_baseline["dental_visit_gap"],
            "routine_checkup_gap_pct_delta_vs_observed": float(scenario_row["routine_checkup_gap"]) - observed_baseline["routine_checkup_gap"],
        }
    )

shelby_scenario_forecast = pd.DataFrame(forecast_rows)
print("Observed Shelby baseline outcomes:", {k: observed_baseline[k] for k in SECONDARY_TARGETS})
print("Model-implied Shelby baseline outcomes:", baseline_model_predictions)
print()
print(shelby_scenario_forecast)


Observed Shelby baseline outcomes: {'obesity_rate': 46.1683489013737, 'diabetes_rate': 21.587827776571586}
Model-implied Shelby baseline outcomes: {'obesity_rate': 46.93361940317903, 'diabetes_rate': 20.19326062920793}

  scenario_name  scenario_step_share                              cluster_id  dental_visit_gap_pct  routine_checkup_gap_pct  obesity_pct_projected  diabetes_pct_projected  obesity_pct_baseline_observed  diabetes_pct_baseline_observed  \
0      baseline                 0.00  Tennessee | Shelby County | cluster_77             55.913424                17.952099              46.168349               21.587828                      46.168349                       21.587828   
1  conservative                 0.15  Tennessee | Shelby County | cluster_77             55.211568                17.899575              46.120839               21.514504                      46.168349                       21.587828   
2      moderate                 0.30  Tennessee | Shelby County | clu

### Cell 14 — Bootstrap uncertainty bands


In [25]:
def extract_fixed_params(pipeline: Pipeline) -> Tuple[float, float]:
    model = pipeline.named_steps["model"]
    return float(model.alpha_), float(model.l1_ratio_)


stage2_fixed_params = {
    target: extract_fixed_params(model)
    for target, model in stage2_models.items()
}

bootstrap_iterations = 200
bootstrap_records = []

stage2_train_frames = {
    target: cluster_ml_sim_table[stage2_features + [target]].copy().dropna(subset=[target]).reset_index(drop=True)
    for target in SECONDARY_TARGETS
}

for bootstrap_id in range(bootstrap_iterations):
    scenario_preds = {}

    for target in SECONDARY_TARGETS:
        train_df = stage2_train_frames[target]
        alpha, l1_ratio = stage2_fixed_params[target]
        sample_idx = np.random.default_rng(42 + bootstrap_id).integers(0, len(train_df), len(train_df))
        sample_df = train_df.iloc[sample_idx].copy().reset_index(drop=True)

        if sample_df[target].nunique(dropna=True) < 5:
            scenario_preds = {}
            break

        X_boot = sample_df[stage2_features].apply(pd.to_numeric, errors="coerce")
        y_boot = pd.to_numeric(sample_df[target], errors="coerce")

        bootstrap_model = make_fixed_elastic_net(alpha=alpha, l1_ratio=l1_ratio)
        bootstrap_model.fit(X_boot, y_boot)

        baseline_boot = clip_pct(
            bootstrap_model.predict(baseline_stage2_features)[0]
        )

        for _, scenario_row in shelby_scenarios.iterrows():
            raw_pred = clip_pct(
                bootstrap_model.predict(pd.DataFrame([scenario_row[stage2_features]]))[0]
            )
            anchored_pred = clip_pct(
                observed_baseline[target] + (raw_pred - baseline_boot)
            )
            scenario_preds[(scenario_row["scenario_name"], target)] = anchored_pred

    for (scenario_name, target), pred in scenario_preds.items():
        bootstrap_records.append(
            {
                "bootstrap_id": bootstrap_id,
                "scenario_name": scenario_name,
                "metric": target,
                "prediction": pred,
            }
        )

bootstrap_predictions = pd.DataFrame(bootstrap_records)

range_rows = []
point_lookup = {
    ("obesity_rate", row["scenario_name"]): row["obesity_pct_projected"]
    for _, row in shelby_scenario_forecast.iterrows()
}
point_lookup.update(
    {
        ("diabetes_rate", row["scenario_name"]): row["diabetes_pct_projected"]
        for _, row in shelby_scenario_forecast.iterrows()
    }
)

for scenario_name, step_share in SCENARIO_STEPS.items():
    for access_target in PRIMARY_TARGETS:
        access_col = "dental_visit_gap_pct" if access_target == "dental_visit_gap" else "routine_checkup_gap_pct"
        point_value = float(
            shelby_scenario_forecast.loc[
                shelby_scenario_forecast["scenario_name"] == scenario_name, access_col
            ].iloc[0]
        )
        range_rows.append(
            {
                "scenario_name": scenario_name,
                "metric": access_target,
                "point_estimate": point_value,
                "p10": point_value,
                "p50": point_value,
                "p90": point_value,
                "n_bootstrap": 0,
                "range_method": "scenario_assumption",
            }
        )

    for target in SECONDARY_TARGETS:
        sub = bootstrap_predictions.loc[
            (bootstrap_predictions["scenario_name"] == scenario_name)
            & (bootstrap_predictions["metric"] == target)
        ].copy()
        point_value = float(point_lookup[(target, scenario_name)])
        if sub.empty:
            p10 = p50 = p90 = np.nan
            n_boot = 0
        else:
            p10 = float(sub["prediction"].quantile(0.10))
            p50 = float(sub["prediction"].quantile(0.50))
            p90 = float(sub["prediction"].quantile(0.90))
            n_boot = int(sub["bootstrap_id"].nunique())

        range_rows.append(
            {
                "scenario_name": scenario_name,
                "metric": target,
                "point_estimate": point_value,
                "p10": p10,
                "p50": p50,
                "p90": p90,
                "n_bootstrap": n_boot,
                "range_method": "bootstrap_fixed_elastic_net_anchored_to_observed_baseline",
            }
        )

shelby_scenario_ranges = pd.DataFrame(range_rows)
print(shelby_scenario_ranges)


   scenario_name               metric  point_estimate        p10        p50        p90  n_bootstrap                                       range_method
0       baseline     dental_visit_gap       55.913424  55.913424  55.913424  55.913424            0                                scenario_assumption
1       baseline  routine_checkup_gap       17.952099  17.952099  17.952099  17.952099            0                                scenario_assumption
2       baseline         obesity_rate       46.168349  46.168349  46.168349  46.168349          200  bootstrap_fixed_elastic_net_anchored_to_observ...
3       baseline        diabetes_rate       21.587828  21.587828  21.587828  21.587828          200  bootstrap_fixed_elastic_net_anchored_to_observ...
4   conservative     dental_visit_gap       55.211568  55.211568  55.211568  55.211568            0                                scenario_assumption
5   conservative  routine_checkup_gap       17.899575  17.899575  17.899575  17.899575        

### Cell 15 — Export slide-ready outputs


In [26]:
outcome_range_lookup = (
    shelby_scenario_ranges.loc[shelby_scenario_ranges["metric"].isin(SECONDARY_TARGETS)]
    .copy()
    .pivot(index="scenario_name", columns="metric", values=["p10", "p90"])
)
outcome_range_lookup.columns = [f"{a}_{b}" for a, b in outcome_range_lookup.columns]
outcome_range_lookup = outcome_range_lookup.reset_index()

shelby_scenario_forecast = shelby_scenario_forecast.merge(outcome_range_lookup, on="scenario_name", how="left")
shelby_scenario_forecast.to_csv(SHELBY_SCENARIO_FORECAST_OUT, index=False)
shelby_scenario_ranges.to_csv(SHELBY_SCENARIO_RANGES_OUT, index=False)

presentation_table = shelby_scenario_forecast[
    [
        "scenario_name",
        "dental_visit_gap_pct",
        "routine_checkup_gap_pct",
        "obesity_pct_projected",
        "diabetes_pct_projected",
        "obesity_pct_delta_vs_observed",
        "diabetes_pct_delta_vs_observed",
        "p10_obesity_rate",
        "p90_obesity_rate",
        "p10_diabetes_rate",
        "p90_diabetes_rate",
    ]
].copy()

presentation_table = presentation_table.rename(
    columns={
        "dental_visit_gap_pct": "Dental visit gap (%)",
        "routine_checkup_gap_pct": "Routine checkup gap (%)",
        "obesity_pct_projected": "Projected obesity (%)",
        "diabetes_pct_projected": "Projected diabetes (%)",
        "obesity_pct_delta_vs_observed": "Obesity delta vs observed (pct pts)",
        "diabetes_pct_delta_vs_observed": "Diabetes delta vs observed (pct pts)",
        "p10_obesity_rate": "Obesity p10",
        "p90_obesity_rate": "Obesity p90",
        "p10_diabetes_rate": "Diabetes p10",
        "p90_diabetes_rate": "Diabetes p90",
    }
)
presentation_table.to_csv(SHELBY_SCENARIO_PRESENTATION_OUT, index=False)

plot_df = shelby_scenario_forecast.copy()

plt.figure(figsize=(10, 5))
x = np.arange(len(plot_df))
width = 0.35
plt.bar(x - width / 2, plot_df["dental_visit_gap_pct"], width=width, label="Dental visit gap")
plt.bar(x + width / 2, plot_df["routine_checkup_gap_pct"], width=width, label="Routine checkup gap")
plt.xticks(x, plot_df["scenario_name"])
plt.ylabel("Percent")
plt.title("Shelby access-gap scenarios")
plt.legend()
plt.tight_layout()
plt.savefig(ACCESS_PLOT_OUT, dpi=200, bbox_inches="tight")
plt.close()

plt.figure(figsize=(10, 5))
x = np.arange(len(plot_df))
width = 0.35
obesity_err = np.vstack(
    [
        plot_df["obesity_pct_projected"] - plot_df["p10_obesity_rate"],
        plot_df["p90_obesity_rate"] - plot_df["obesity_pct_projected"],
    ]
)
diabetes_err = np.vstack(
    [
        plot_df["diabetes_pct_projected"] - plot_df["p10_diabetes_rate"],
        plot_df["p90_diabetes_rate"] - plot_df["diabetes_pct_projected"],
    ]
)
plt.bar(x - width / 2, plot_df["obesity_pct_projected"], width=width, yerr=obesity_err, capsize=4, label="Projected obesity")
plt.bar(x + width / 2, plot_df["diabetes_pct_projected"], width=width, yerr=diabetes_err, capsize=4, label="Projected diabetes")
plt.xticks(x, plot_df["scenario_name"])
plt.ylabel("Percent")
plt.title("Shelby downstream outcome projections")
plt.legend()
plt.tight_layout()
plt.savefig(OUTCOME_PLOT_OUT, dpi=200, bbox_inches="tight")
plt.close()

print("Exported files:")
for path in [
    TRACT_HEALTH_TARGETS_OUT,
    CLUSTER_HEALTH_TARGETS_OUT,
    CLUSTER_ML_SIM_TABLE_OUT,
    STAGE1_METRICS_OUT,
    STAGE2_METRICS_OUT,
    SHELBY_SCENARIO_FORECAST_OUT,
    SHELBY_SCENARIO_RANGES_OUT,
    SHELBY_SCENARIO_PRESENTATION_OUT,
    ACCESS_PLOT_OUT,
    OUTCOME_PLOT_OUT,
]:
    print("-", path)


Exported files:
- C:\Users\jabba\Desktop\Code\machine_learning\AUC_mastercard_challenge\src\data\processed\ml_simulation\tract_health_targets.parquet
- C:\Users\jabba\Desktop\Code\machine_learning\AUC_mastercard_challenge\src\data\processed\ml_simulation\cluster_health_targets.parquet
- C:\Users\jabba\Desktop\Code\machine_learning\AUC_mastercard_challenge\src\data\processed\ml_simulation\cluster_ml_sim_table.parquet
- C:\Users\jabba\Desktop\Code\machine_learning\AUC_mastercard_challenge\src\data\processed\ml_simulation\stage1_model_metrics.csv
- C:\Users\jabba\Desktop\Code\machine_learning\AUC_mastercard_challenge\src\data\processed\ml_simulation\stage2_model_metrics.csv
- C:\Users\jabba\Desktop\Code\machine_learning\AUC_mastercard_challenge\src\data\processed\ml_simulation\shelby_scenario_forecast.csv
- C:\Users\jabba\Desktop\Code\machine_learning\AUC_mastercard_challenge\src\data\processed\ml_simulation\shelby_scenario_ranges.csv
- C:\Users\jabba\Desktop\Code\machine_learning\AUC_mas

### Cell 16 — Interpretation


In [27]:
baseline_row = shelby_scenario_forecast.loc[shelby_scenario_forecast["scenario_name"] == "baseline"].iloc[0]
aggressive_row = shelby_scenario_forecast.loc[shelby_scenario_forecast["scenario_name"] == "aggressive"].iloc[0]

print("Interpretation notes")
print("- These are peer-informed predictive scenarios, not causal estimates.")
print("- Stage 1 models learn cluster-level access patterns; Stage 2 models translate access-gap changes into slower-moving health outcomes.")
print("- The scenario lever is conservative by design: Shelby only moves partway toward access outcomes already observed in better-performing similar peers.")
print(
    f"- In the aggressive scenario, dental visit gap improves by "
    f"{baseline_row['dental_visit_gap_pct'] - aggressive_row['dental_visit_gap_pct']:.2f} pct pts and "
    f"routine checkup gap improves by "
    f"{baseline_row['routine_checkup_gap_pct'] - aggressive_row['routine_checkup_gap_pct']:.2f} pct pts."
)
print(
    f"- The model projects obesity to move from {baseline_row['obesity_pct_baseline_observed']:.2f}% "
    f"to {aggressive_row['obesity_pct_projected']:.2f}% and diabetes from "
    f"{baseline_row['diabetes_pct_baseline_observed']:.2f}% to {aggressive_row['diabetes_pct_projected']:.2f}% "
    f"under that aggressive-but-still-partial peer catch-up scenario."
)
print("- Puerto Rico clusters remain unmatched because the CDC PLACES tract endpoint used here does not cover them in this cache.")


Interpretation notes
- These are peer-informed predictive scenarios, not causal estimates.
- Stage 1 models learn cluster-level access patterns; Stage 2 models translate access-gap changes into slower-moving health outcomes.
- The scenario lever is conservative by design: Shelby only moves partway toward access outcomes already observed in better-performing similar peers.
- In the aggressive scenario, dental visit gap improves by 2.11 pct pts and routine checkup gap improves by 0.16 pct pts.
- The model projects obesity to move from 46.17% to 46.03% and diabetes from 21.59% to 21.37% under that aggressive-but-still-partial peer catch-up scenario.
- Puerto Rico clusters remain unmatched because the CDC PLACES tract endpoint used here does not cover them in this cache.
